# SOLiD-PINN v0.8.4 — Automated Colab → GitHub

**One-time setup:** Create a fine-grained GitHub personal access token restricted to `theerayutbo/SOLiD-PINN` with **Contents: Read and write** (Metadata read is automatic). Use an expiry date. In Colab's key-shaped Secrets panel, create `SOLID_PINN_GITHUB_TOKEN` and enable Notebook access. Do not paste the token in code, chat or output.

Select CPU and Run all. Mount Drive when prompted. The notebook checks out pinned code, runs tests, runs the full 108-posterior synthetic benchmark, validates complete results and automatically publishes only a whitelisted aggregate JSON to a separate content-addressed branch. No private inputs are read. No write to main, no merge, no overwrite of prior results. The full synthetic report is saved to Drive before the publication cell. If authorization fails, correct it and rerun only the final cell. Publication is verified against the remote bytes. This is an analytical Bayesian reference, not Bayesian PINN or experimental validation.

In [ ]:
import os, sys, subprocess, tempfile, json, hashlib, uuid
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount("/content/drive")
REV="1b4223f43aeab236a9868ed71c5bb46c21b7c028"
ROOT=Path("/content/drive/MyDrive/SOLiD-PINN/colab_runs/v084")
RUN=ROOT/(datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")+"-"+uuid.uuid4().hex[:8])
RUN.mkdir(parents=True,exist_ok=False)
REPO=Path(tempfile.mkdtemp(prefix="solid-pinn-v084-"))/"repo"
subprocess.run(["git","clone","--no-checkout","https://github.com/theerayutbo/SOLiD-PINN.git",str(REPO)],check=True)
subprocess.run(["git","checkout","--detach",REV],cwd=REPO,check=True)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=REPO,text=True).strip()==REV
env=os.environ.copy()
env.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",NUMEXPR_NUM_THREADS="1")
subprocess.run([sys.executable,"-m","pip","install","numpy==2.3.5","scipy==1.17.0","pytest","requests"],check=True)
sys.path.insert(0,str(REPO/"scripts"))
import publish_colab_uq as pub
solver=(REPO/"scripts/bayes_interface.py").read_bytes()
tests=(REPO/"scripts/test_bayes_interface.py").read_bytes()
assert pub.git_blob(solver)==pub.SOURCE_BLOB
assert pub.git_blob(tests)==pub.TEST_BLOB
subprocess.run([sys.executable,"-m","pytest","-q","scripts/test_bayes_interface.py","scripts/test_publish_colab_uq.py"],cwd=REPO,env=env,check=True)
SETUP_OK=True
print("Pinned revision:",REV,"\nRun directory:",RUN)

## Full Bayesian reference run
The run is CPU-only. The existing solver has no intermediate checkpoint, so an interrupted computation must be restarted. Completed results are persisted before publication. Negative scientific findings are preserved, not filtered out.

In [ ]:
assert SETUP_OK
REPORT=RUN/"v084_full.json"
BUNDLE=RUN/"v084_summary.json"
if REPORT.exists() or BUNDLE.exists(): raise RuntimeError("Run directory already contains output; use a new run or rerun publication only")
with tempfile.TemporaryDirectory() as tmp:
    output=Path(tmp)/"v084_full.json"
    subprocess.run([sys.executable,"scripts/bayes_interface.py","--out",str(output),"--n","151","--draws","16000","--repeats","30"],cwd=REPO,env=env,check=True)
    raw=output.read_bytes()
    bundle=pub.build_bundle(raw,REV,solver,tests)
    for dst,data in [(REPORT,raw),(BUNDLE,pub.canonical(bundle))]:
        part=dst.with_suffix(dst.suffix+".part")
        part.write_bytes(data)
        part.replace(dst)
print("Completed:",bundle["completed"])
print("Coverage:",json.dumps(bundle["coverage"],indent=2))
print("Saved full report and public summary:",RUN)

## Publish verified results to GitHub
Only the validated aggregate summary is uploaded. The token is retrieved here, after computation, and is never stored in the report or passed as a command-line argument. If this cell fails, no success is claimed; the report remains in Drive. Rerun this cell after fixing authorization.

In [ ]:
from google.colab import userdata
# Revalidate on every retry; do not publish an edited or incomplete summary.
bundle=pub.build_bundle(REPORT.read_bytes(),REV,solver,tests)
assert pub.canonical(bundle)==BUNDLE.read_bytes()
token=userdata.get("SOLID_PINN_GITHUB_TOKEN")
try:
    receipt=pub.publish(bundle,token)
finally:
    del token
(RUN/"publication_receipt.json").write_bytes(pub.canonical(receipt))
print("REMOTE VERIFIED:",receipt["url"])
print("SHA256:",receipt["sha256"])
from IPython.display import display, Markdown
display(Markdown("[Open verified GitHub results]("+receipt["url"]+")"))